# Policy decision ensemble — Databricks round 2 (league bot)

Implements prompts R0–R4 of `databricks/ENSEMBLE_PLAN.md` section 9: relabel with the **league** bot itself
playing (v3 rules + `spoc_db` Q-net at 0.85 + the round-1 ensemble as a correction at 0.5), with
`pool/qnet1` in the opponent pool. Round-1 rows were labelled by the v3-only bot, so round 2 trains on
`policy_rows_r2` **only**. Same method as round 1 (reference `train/policy_train.py`).

Round-2 results (data `workspace.default.policy_rows_r2`, 200,462 rows from seeds L2-1..L2-3100, k=8):

| stage | outcome |
|---|---|
| R0 setup | `qnet_weight` 0.85; pool (15) includes `pool/qnet1`; league vs qnet1 +3.98 bb/deal (100 deals); selftest 19 rows, 0.9 s |
| R1 data | 200,462 rows / 3,100 units in ~49 min (~11 core-hours, league policy is ~40% heavier per playout) |
| R2 quality | 0 broken rows; advantage SD 20–26 chips; FOLD best on 43–54% of facing-raise spots (league calls more than v3 did) |
| R3 train | ensemble adv MAE 12.39, best-action 35.9%, offline value w=0: +4.72 → w=1: +5.69 chips/decision (MLflow `poker_policy_ensemble`, runs `r2_*`) |
| R4 export | `bot/policy_weights.npz` (ridge + mlp0..4); fresh-process parity max|diff| = 0 over 1,000 rows; predict 0.046 ms p50 / 0.048 ms p95 |
| handover | branch `db-league-r2` |

In [0]:
# R0: setup check on the league branch.
# The kernel is long-lived: after any pull or branch switch, purge repo modules before importing.
REPO = "/Workspace/Users/<your-databricks-user>/datathon"  # placeholder: set to your repo copy
import sys, time
for m in [m for m in list(sys.modules)
          if m.split(".")[0] in ("bot", "train", "selfplay", "tune", "opponents", "evaluator", "game",
                                 "strategy", "policy_data", "policy_features", "policy_model", "opponent",
                                 "equity", "equity_model", "compare", "fit_preflop", "policy_train", "qnet", "q_features")]:
    del sys.modules[m]
sys.path.insert(0, REPO)
sys.path.insert(0, REPO + "/train")
sys.path.insert(0, REPO + "/baseline-bots")
from evaluator import warmup
from bot import strategy
from train import selfplay, policy_data
warmup()

P = strategy.load_params()
assert P["qnet_weight"] == 0.85
assert "pool/qnet1" in policy_data.pool()
mean, lo, hi = selfplay.match(P, selfplay.opponent_pool()["pool/qnet1"], 100, "check")
print(f"league vs pool/qnet1 (100 deals): {mean:+.3f} [{lo:+.3f}, {hi:+.3f}] bb/deal")
t0 = time.perf_counter()
print("selftest rows:", policy_data.selftest(), f"in {time.perf_counter()-t0:.1f} s")

In [0]:
# R1: one work unit = generate_labeled(seed, opp, n_hands=200, k=8) with the LEAGUE bot replaying.
# Opponents drawn by weight from policy_data.pool() (15 members incl. pool/qnet1) with driver RNG "db-draw-L2".
import random
import pandas as pd

pool = policy_data.pool()
names, weights = zip(*[(n, pool[n][1]) for n in pool])
TOTAL = 3100  # ~200k rows at ~64.7 rows/unit (observed)
rng = random.Random("db-draw-L2")
opps = rng.choices(list(names), weights=list(weights), k=TOTAL)

def run_units(batches):
    import sys
    import pandas as pd
    REPO = "/Workspace/Users/<your-databricks-user>/datathon"
    sys.path.insert(0, REPO)
    sys.path.insert(0, REPO + "/train")
    sys.path.insert(0, REPO + "/baseline-bots")
    from train import policy_data
    from evaluator import warmup
    warmup()
    for pdf in batches:
        rows = []
        for r in pdf.itertuples(index=False):
            for row in policy_data.generate_labeled(r.seed, r.opp, n_hands=200, k=8):
                rows.append((row["seed"], row["opponent"], int(row["hand_index"]), row["street"],
                             int(row["facing_raise"]), int(row["pot"]),
                             [float(x) for x in row["features"]],
                             [float(x) for x in row["ev"]],
                             [float(x) for x in row["q"]],
                             [float(x) for x in row["q_std"]]))
        yield pd.DataFrame(rows, columns=["seed", "opponent", "hand_index", "street", "facing_raise",
                                         "pot", "features", "ev", "q", "q_std"])

def write_batch(lo, hi, mode):
    spec = pd.DataFrame({"seed": [f"L2-{i+1}" for i in range(lo, hi)], "opp": opps[lo:hi]})
    (spark.createDataFrame(spec).repartition(min(220, hi - lo))
          .mapInPandas(run_units,
                       "seed string, opponent string, hand_index int, street string, facing_raise int, pot int, "
                       "features array<double>, ev array<double>, q array<double>, q_std array<double>")
          .write.mode(mode).format("delta").saveAsTable("workspace.default.policy_rows_r2"))
    print(lo + 1, "..", hi, "->", spark.read.table("workspace.default.policy_rows_r2").count(), "rows")

In [0]:
# R1 pilot: 20 units, 4 partitions (NOT persisted — L2-1..L2-20 are re-run and written below).
# Observed: 1,448 rows in 63 s -> 3.1 s/unit, 72.4 rows/unit (league is ~40% heavier per unit than v3-only).
t0 = time.perf_counter()
pilot_rows = (spark.createDataFrame(pd.DataFrame({"seed": [f"L2-{i+1}" for i in range(20)], "opp": opps[:20]}))
       .repartition(4)
       .mapInPandas(run_units,
                    "seed string, opponent string, hand_index int, street string, facing_raise int, pot int, "
                    "features array<double>, ev array<double>, q array<double>, q_std array<double>")).count()
print(f"pilot: 20 units -> {pilot_rows} rows in {time.perf_counter()-t0:.0f} s")

In [0]:
# R1 full run: 3,100 units in 3 batches (wall times: 955 s + 965 s + 1,050 s, ~49 min total).
# Rows/unit settled at ~64 after the pilot, so TOTAL was raised from 2760 to 3100 to hit ~200k rows;
# the driver RNG stream keeps the first draws identical.
write_batch(0, 1000, "overwrite")
write_batch(1000, 2000, "append")
write_batch(2000, 3100, "append")

In [0]:
%sql
-- Rows by street and facing_raise
SELECT street, COUNT(*) n, SUM(facing_raise) facing_raise_rows, ROUND(AVG(pot), 1) avg_pot
FROM workspace.default.policy_rows_r2 GROUP BY street
ORDER BY CASE street WHEN 'PREFLOP' THEN 0 WHEN 'FLOP' THEN 1 WHEN 'TURN' THEN 2 ELSE 3 END;

In [0]:
%sql
-- Rows by opponent (15 pool members; pool/qnet1 = spoc_db's bot, new in round 2)
SELECT opponent, COUNT(*) n, ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) pct
FROM workspace.default.policy_rows_r2 GROUP BY opponent ORDER BY n DESC;

In [0]:
# R2: broken rows, advantage stats, FOLD-best facing a raise, noise floor on policy_rows_r2.
# Round-2 findings: 0 broken rows; advantage SD 19.6-26.0 chips; FOLD mean advantage -6.0 (worse than
# round 1's -4.3: under league continuations folding is even less attractive); FOLD best on 43-54% of
# facing-raise spots; noise floor 0.7-3.7 chips; corr(EV,Q)=0.295.
import math
import numpy as np

pdf = spark.read.table("workspace.default.policy_rows_r2").toPandas()
sys.path.insert(0, REPO)
from game import Action
ACTIONS = [a.name for a in Action]  # FOLD, CHECK_CALL, RAISE_HALF, RAISE_POT
n = len(pdf)
Q = np.array([[float(x) for x in r] for r in pdf.q])
EV = np.array([[float(x) for x in r] for r in pdf.ev])
QS = np.array([[float(x) for x in r] for r in pdf.q_std])
legal = ~np.isnan(Q)
broken = int((~legal[:, 1]).sum() + (legal.sum(axis=1) == 0).sum()
             + pdf.pot.isna().sum() + (pdf.features.apply(len) != 30).sum())
print(f"rows {n}, broken {broken} ({broken / n * 100:.2f}%) -> {'STOP' if broken > 0.01 * n else 'OK'}")

adv = Q - np.where(legal, np.nanmean(np.where(legal, Q, np.nan), axis=1, keepdims=True), np.nan)
for i, name in enumerate(ACTIONS):
    v = adv[:, i][legal[:, i]]
    print(f"adv {name:10s} n={len(v):6d}  mean {v.mean():+7.2f}  sd {v.std():6.2f}")
for st in ("PREFLOP", "FLOP", "TURN", "RIVER"):
    m = (pdf.street == st).values
    v = adv[m].ravel(); v = v[~np.isnan(v)]
    nf = (QS[m] / math.sqrt(8)).ravel(); nf = nf[~np.isnan(nf)]
    mr = ((pdf.street == st) & (pdf.facing_raise == 1)).values
    best = np.nanargmax(np.where(legal & mr[:, None], Q, -np.inf), axis=1)[mr]
    print(f"{st:8s} adv mean {v.mean():+5.1f} sd {v.std():5.1f} | noise {nf.mean():4.1f} | FOLD best when raised {np.mean(best == 0) * 100:4.1f}%")

In [0]:
# R3: train on policy_rows_r2 ONLY (round-1 rows were labelled by the v3-only bot — stale values).
# train/policy_train.py functions used UNCHANGED; split by seed (zlib.crc32%5==0).
# Round-2: train 161,174 / test 39,288; ridge ~0 s, each MLP ~65 s.
import zlib, shutil
for m in [m for m in list(sys.modules) if m.split(".")[0] in ("bot", "train", "selfplay", "tune", "opponents",
                                                            "strategy", "policy_data", "policy_train",
                                                            "evaluator", "game", "opponent", "qnet", "q_features")]:
    del sys.modules[m]
sys.path.insert(0, REPO + "/train")
import policy_train as pt
warmup()

F = np.array([list(map(float, r)) for r in pdf.features])
POT = pdf.pot.to_numpy(float)
SEED = np.array([str(s) for s in pdf.seed])
mask = ~np.isnan(Q)
adv = np.where(mask, Q - np.nanmean(np.where(mask, Q, np.nan), axis=1, keepdims=True), 0.0)
test = np.array([zlib.crc32(s.encode()) % 5 == 0 for s in SEED])
mu, sd = F[~test].mean(0), F[~test].std(0); sd[sd < 1e-6] = 1.0
Z = (F - mu) / sd
tr = ~test
temp = pt.strategy.load_params()["temperature"]
print(f"train rows {tr.sum()}, test rows {test.sum()} (split by seed)")

members = {"lin": pt.ridge(Z[tr], adv[tr], mask[tr])}
for i in range(5):
    members[f"mlp{i}"] = pt.mlp(Z[tr], adv[tr], mask[tr], seed=i + 1)
preds = {k: pt.forward(v, Z[test]) for k, v in members.items()}
ens = np.mean(list(preds.values()), axis=0)
d = {"q": Q, "ev": EV, "pot": POT, "seed": SEED}
for name, pred in {**preds, "ensemble": ens}.items():
    pt.report(name, pred, adv[test], mask[test], d, test, temp)

In [0]:
# R3 MLflow: experiment poker_policy_ensemble, run names r2_lin / r2_mlp0..4 / r2_ensemble.
import mlflow
mlflow.autolog(disable=True)
mlflow.set_experiment("poker_policy_ensemble")
tv = int(spark.sql("DESCRIBE HISTORY workspace.default.policy_rows_r2").first()["version"])
for name, pred in {**preds, "ensemble": ens}.items():
    mae = np.abs(pred - adv[test])[mask[test]].mean()
    big = (np.where(mask[test], pred, -np.inf).argmax(1)
           == np.where(mask[test], adv[test], -np.inf).argmax(1)).mean()
    rows = pt.rows_of(d, test)
    with mlflow.start_run(run_name=f"r2_{name}"):
        mlflow.log_param("member", name)
        mlflow.log_param("round", 2)
        mlflow.log_param("data_table", "workspace.default.policy_rows_r2")
        mlflow.log_param("table_version", tv)
        mlflow.log_param("train_rows", int(tr.sum()))
        mlflow.log_param("test_rows", int(test.sum()))
        mlflow.log_metric("adv_mae", float(mae))
        mlflow.log_metric("best_action_match", float(big))
        for w in (0.0, 0.25, 0.5, 1.0, 2.0):
            mlflow.log_metric(f"offline_value_w{w}", float(pt.offline_value(rows, pred, w, temp)))

In [0]:
# R3 ensemble latency, single row, single CPU: round-2 p50 0.064 ms / p95 0.065 ms.
def predict_one(f):
    x = (np.asarray(f, dtype=np.float64) - mu) / sd
    o = np.zeros(4)
    for layers in members.values():
        a = x
        for i, (W_, b_) in enumerate(layers):
            a = a @ W_ + b_
            if i < len(layers) - 1:
                a = np.maximum(a, 0.0)
        o += a / len(members)
    return o
ts = []
for f in F[test][:500]:
    t0 = time.perf_counter(); predict_one(f); ts.append(time.perf_counter() - t0)
print(f"ensemble latency: p50={np.median(ts)*1000:.3f} ms  p95={np.percentile(ts, 95)*1000:.3f} ms")

In [0]:
# R4: export bot/policy_weights.npz (r2 members + mu/sd; bot/policy_model.py unchanged) and
# bot/policy_parity.npz (1,000 held-out test rows). Fresh-process check in a fresh executor worker.
# Round-2: parity max|diff| = 0 over 1,000 rows; predict 0.046 ms p50 / 0.048 ms p95.
out = {"mu": mu, "sd": sd, "members": np.array(list(members)), "alpha": np.ones(len(members))}
for k, layers in members.items():
    for i, (W_, b_) in enumerate(layers):
        out[f"{k}/W{i}"], out[f"{k}/b{i}"] = W_, b_
np.savez("/tmp/policy_weights_r2.npz", **out)

sel = np.where(test)[0][:1000]
Zp = (F[sel] - mu) / sd
exp = np.mean([pt.forward(v, Zp) for v in members.values()], axis=0)
np.savez("/tmp/policy_parity_r2.npz", features=F[sel], expected=exp)
shutil.copy2("/tmp/policy_weights_r2.npz", REPO + "/bot/policy_weights.npz")
shutil.copy2("/tmp/policy_parity_r2.npz", REPO + "/bot/policy_parity.npz")

par = pd.DataFrame({"features": [F[i] for i in sel], "expected": [exp[i] for i in range(len(sel))]})
def check(batches):
    import sys, time
    import numpy as np
    sys.path.insert(0, "/Workspace/Users/<your-databricks-user>/datathon")
    from bot import policy_model  # fresh import: loads the exported r2 weights
    for p_ in batches:
        maxdiff, t_ = 0.0, []
        for f, e in zip(p_.features, p_.expected):
            t0 = time.perf_counter(); o = policy_model.predict(f); t_.append(time.perf_counter() - t0)
            maxdiff = max(maxdiff, float(np.max(np.abs(np.asarray(o) - np.asarray(e)))))
        yield pd.DataFrame([{"max_diff": maxdiff, "p50_ms": float(np.median(t_)) * 1000,
                             "p95_ms": float(np.percentile(t_, 95)) * 1000, "n": len(t_)}])
res = (spark.createDataFrame(par).repartition(1)
       .mapInPandas(check, "max_diff double, p50_ms double, p95_ms double, n int").toPandas())
print(res.to_string(index=False))

## R4: handover

`bot/policy_weights.npz`, `bot/policy_parity.npz` and this notebook are committed to branch
**`db-league-r2`** (from `league`; no other files changed; never force-pushed). `policy_w` stays at 0.5
until the local gate passes: parity test, `train/compare.py` vs `league`, then 50 HTTP matches vs
`league` and 25 vs `spoc_db` (per ENSEMBLE_PLAN.md section 9).

MLflow experiment `poker_policy_ensemble` (round 2, data table `workspace.default.policy_rows_r2`):
lin `ec505a9b62e149b183381ba23f6a9bcf`, mlp0 `8ff0856ae19c4cd7a8c17b473bc58aa9`,
mlp1 `865cd89ee7024b579102f63a3e42e7af`, mlp2 `b6eb024077e546869e9590409b360a9f`,
mlp3 `85a5f96131c3479b8727fab0a2930698`, mlp4 `490eef8877f44d238319bc8cd3983a43`,
ensemble `71a796bca3b042c5853c4f7b080d2907`.

Offline value on held-out seeds (chips/decision, `policy_data.offline_value`): rules-only w=0 +4.719,
w=0.25 +5.220, w=0.5 +5.471, **w=1 +5.692**, w=2 +5.834 — the correction dominates the rules at every
tested weight on league-reached states (note: these labels and playouts come from the full league bot).